# Cell 1: Imports

In [1]:
import sagemaker
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    InputData, 
    S3DataSource, 
    SourceCode, 
    Compute,
    OutputDataConfig,
    StoppingCondition,
    MetricDefinition,
    StoppingCondition
)

[02/04/26 12:33:37] INFO     Found credentials in environment variables.                        ]8;id=586817;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=113822;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py#1252\1252]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
# --- USER SETTINGS ---
my_job_name = "2DCNN-LSTM-Exp-2nd_1000_resume" 
my_bucket = "alexander-thesis-cslr"

In [3]:
# 1. Metric Definitions (The Magic Part) 🔍
# Adjust the 'Regex' to match EXACTLY what your train_sagemaker.py prints.
# Example assumption: Your script prints "Epoch: 1, Loss: 0.45, Acc: 0.82"
# Updated Metric Definitions (using r'' to fix SyntaxWarning)
metrics = [
    MetricDefinition(name='train:loss',  regex=r'\[Metrics\] Train Loss: ([0-9\.]+)'),
    MetricDefinition(name='train:top1',  regex=r'\[Metrics\] Train Top1: ([0-9\.]+)'),
    MetricDefinition(name='train:top5',  regex=r'\[Metrics\] Train Top5: ([0-9\.]+)'),
    MetricDefinition(name='train:top10', regex=r'\[Metrics\] Train Top10: ([0-9\.]+)'),
    MetricDefinition(name='val:loss',    regex=r'\[Metrics\] Val Loss: ([0-9\.]+)'),
    MetricDefinition(name='val:top1',    regex=r'\[Metrics\] Val Top1: ([0-9\.]+)'),
    MetricDefinition(name='val:top5',    regex=r'\[Metrics\] Val Top5: ([0-9\.]+)'),
    MetricDefinition(name='val:top10',   regex=r'\[Metrics\] Val Top10: ([0-9\.]+)')
]

In [4]:
# 1. Define your code location
# source_dir='.' works because your notebook is inside Thesis-CSLR
code_config = SourceCode(
    source_dir='./ASL',
    entry_script='train_sagemaker.py' 
)

In [5]:
# 3. Hardware Configuration
# 'instance_type' and count move to this new object
compute_config = Compute(
    instance_type="ml.g5.xlarge",
    instance_count=1
)

# Cell 2: Setup

In [6]:
# 4. Data Config
data_source = S3DataSource(
    s3_uri=f"s3://{my_bucket}/data_tensors_1000",
    s3_data_type="S3Prefix",
    s3_data_distribution_type="FullyReplicated"
)

In [7]:
data_input = InputData(
    channel_name="training",
    data_source=data_source
)

In [8]:
# --- 1. Define Resume Source ---
# Point to the FOLDER (prefix) containing model.tar.gz
resume_s3_uri = "s3://alexander-thesis-cslr/experiments/output/Thesis-ASL-Exp-2DCNN-LSTM-1st-1000-20260203185308/output/"

resume_source = S3DataSource(
    s3_uri=resume_s3_uri,
    s3_data_type="S3Prefix",
    s3_data_distribution_type="FullyReplicated"
)

resume_input = InputData(
    channel_name="resume",  # <--- script looks for os.environ.get("SM_CHANNEL_RESUME")
    data_source=resume_source
)

In [9]:
# 5. Output Config
output_config = OutputDataConfig(
    s3_output_path=f"s3://{my_bucket}/experiments/output"
)

In [10]:
stop_condition = StoppingCondition(
    max_runtime_in_seconds=432000
)

# Cell 3: Define the Experiment

In [11]:
# 3. Initialize the Unified Trainer
# You now provide the image URI directly
trainer = ModelTrainer(
    training_image="763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-training:2.0-gpu-py310",
    role="arn:aws:iam::600889066998:role/SageMaker_role_Alexander_for_thesis_CLSR",
    base_job_name="Thesis-ASL-Exp-2DCNN-LSTM-1st_1000",
    source_code=code_config,      # script location
    compute=compute_config,
    output_data_config=output_config,
    hyperparameters={
        "epochs": "200",
        "batch-size": "22",
        "num-classes": "1000",
        "learning-rate": "0.0003",
        "experiment-name": my_job_name,
        "model-type" : "2dcnn_lstm",
        "start-epoch" : "11"
    },
    # Set FastFile globally for the algorithm
    training_input_mode="File",
    environment={"PYTHONUNBUFFERED": "1"},
    stopping_condition=stop_condition,
    tags=[
        {'key': 'Project', 'value': 'CSLR-Thesis'},
        {'key': 'Model',   'value': '2DCNN-LSTM'},
        {'key': 'User',    'value': 'Alexander'}
    ],
    
).with_metric_definitions(metrics)


[02/04/26 12:33:38] INFO     Found credentials in environment variables.                        ]8;id=563584;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=317876;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py#1252\1252]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=604596;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=13282;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=111800;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=674385;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#162\162]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=905629;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=575008;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#548\548]8;;\
                             763104351884.dkr.ecr.eu-north-1.amazonaws.com/pytorch-training:2.                     
                             0-gpu-py310                                                                           

# Cell 4: Launch!

In [ ]:
print(f"🚀 Launching Job with Monitoring: {my_job_name}")
training_job = trainer.train(
    input_data_config=[data_input, resume_input],
    wait=True
)

print(f"✅ Job submitted! Check the 'Monitor' tab in the console.")

🚀 Launching Job with Monitoring: 2DCNN-LSTM-Exp-2nd_1000_resume


                    INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=747186;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=881153;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[02/04/26 12:33:42] INFO     Creating training_job resource.                                     ]8;id=466278;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=480977;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#35539\35539]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=152008;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=12584;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#340\340]8;;\

                    INFO     Runs on sagemaker prod, region:eu-north-1                                 ]8;id=691764;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=758397;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#354\354]8;;\

                    INFO     Found credentials in environment variables.                        ]8;id=736062;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=572376;file:///home/sagemaker-user/Thesis-CSLR/.venv/lib/python3.12/site-packages/botocore/credentials.py#1252\1252]8;;\

Output()